# YgoMaster-PackBuilder — YGOPRODeck Live Client Test

Notebook di integrazione **live** per il custom pack YGOPRODeck:

```text
external_id = "24116"
pack = "Scaled Invasion"
```

Questo notebook effettua richieste reali di rete.

Obiettivi:

- verificare che `YgoProDeckClient` raggiunga l'endpoint;
- ispezionare la risposta grezza;
- verificare la forma `list[dict]`;
- controllare alcuni campi attesi;
- eseguire il flusso completo `fetch -> parse`;
- verificare i metadata principali del pack;
- controllare cardinalità, ID e rarità sorgente senza applicare ancora regole di dominio.

> Se questo notebook fallisce per timeout, DNS, HTTP 4xx/5xx o cambiamenti
> dell'endpoint, il parser offline può comunque continuare a funzionare correttamente.


## 1. Ambiente e root repository

In [18]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository."
    )

print("Repository root:", repo_root)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder


## 2. Import

In [19]:
from collections import Counter

from ygomaster_packbuilder.sources.ygoprodeck import (
    YgoProDeckClient,
    YgoProDeckPackData,
    parse_custom_pack,
)

print("Imports OK")

Imports OK


## 3. Configurazione del test live

In [20]:
PACK_ID = "24116"
EXPECTED_PACK_NAME = "Scaled Invasion"

client = YgoProDeckClient(timeout=15.0)

print("Pack ID:", PACK_ID)
print("Expected name:", EXPECTED_PACK_NAME)

Pack ID: 24116
Expected name: Scaled Invasion


## 4. Fetch live della risposta grezza

Questa è la prima cella che effettua una richiesta HTTP reale.


In [21]:
raw = client.fetch_custom_pack_raw(PACK_ID)

print("Raw response type:", type(raw).__name__)
print("Rows received:", len(raw))

assert isinstance(raw, list)
assert len(raw) > 0

print("Live fetch OK")

Raw response type: list
Rows received: 100
Live fetch OK


## 5. Ispezione della prima riga

In [22]:
first = raw[0]

assert isinstance(first, dict)

print("Available keys:")
for key in sorted(first.keys()):
    print(" -", key)

print()
print("First row:")
first

Available keys:
 - card_rarity
 - code
 - id
 - img_url
 - name
 - pack_description
 - pack_name
 - pack_views
 - type
 - username

First row:


{'username': 'livius03',
 'pack_name': 'Scaled Invasion',
 'pack_description': 'A pack centered around the Venom and Alien Archetypes, with extra support from Reptilianne, Ogdoadic, and generic Reptile-focused cards.',
 'img_url': '8400623',
 'code': 501,
 'name': '"A" Cell Breeding Device',
 'type': 'Continuous Spell',
 'id': 34541863,
 'pack_views': 0,
 'card_rarity': 'Common'}

## 6. Campi minimi attesi

Non controlliamo ancora i valori semantici: verifichiamo solo che l'endpoint
fornisca i campi di cui il parser ha bisogno.


In [23]:
required_fields = {
    "username",
    "pack_name",
    "pack_description",
    "img_url",
    "name",
    "type",
    "id",
    "card_rarity",
}

missing = required_fields - set(first.keys())

assert not missing, (
    f"YGOPRODeck response is missing required fields: {sorted(missing)}"
)

print("Required fields present")

Required fields present


## 7. Metadata grezzi del pack

In [24]:
print("Creator:", first["username"])
print("Pack name:", first["pack_name"])
print("Description:", first["pack_description"])
print("Image ID:", first["img_url"])

assert first["pack_name"] == EXPECTED_PACK_NAME

print("Raw pack metadata OK")

Creator: livius03
Pack name: Scaled Invasion
Description: A pack centered around the Venom and Alien Archetypes, with extra support from Reptilianne, Ogdoadic, and generic Reptile-focused cards.
Image ID: 8400623
Raw pack metadata OK


## 8. Controllo coerenza metadata nella risposta grezza

Il parser farà lo stesso tipo di controllo in modo formale.
Qui lo visualizziamo per il test live.


In [25]:
metadata_fields = (
    "username",
    "pack_name",
    "pack_description",
    "img_url",
)

reference = {
    field: first[field]
    for field in metadata_fields
}

inconsistent_rows = []

for index, row in enumerate(raw):
    for field in metadata_fields:
        if row.get(field) != reference[field]:
            inconsistent_rows.append(
                (index, field, row.get(field), reference[field])
            )

assert not inconsistent_rows, inconsistent_rows

print("All rows share consistent pack metadata")

All rows share consistent pack metadata


## 9. Parsing della risposta live

In [26]:
pack = parse_custom_pack(
    external_id=PACK_ID,
    data=raw,
)

assert isinstance(pack, YgoProDeckPackData)

print("Live response parsed successfully")
pack

Live response parsed successfully


YgoProDeckPackData(external_id='24116', name='Scaled Invasion', description='A pack centered around the Venom and Alien Archetypes, with extra support from Reptilianne, Ogdoadic, and generic Reptile-focused cards.', creator='livius03', image_id='8400623', cards=[YgoProDeckPackCard(name='"A" Cell Breeding Device', ydk_id=34541863, card_type='Continuous Spell', source_rarity='Common'), YgoProDeckPackCard(name='"A" Cell Incubator', ydk_id=64163367, card_type='Continuous Spell', source_rarity='Common'), YgoProDeckPackCard(name='Attack Pheromones', ydk_id=68786330, card_type='Continuous Spell', source_rarity='Common'), YgoProDeckPackCard(name='Code A Ancient Ruins', ydk_id=99342953, card_type='Continuous Spell', source_rarity='Common'), YgoProDeckPackCard(name="Snake Deity's Command", ydk_id=80678380, card_type='Counter Trap', source_rarity='Common'), YgoProDeckPackCard(name='Alien Dog', ydk_id=15475415, card_type='Effect Monster', source_rarity='Common'), YgoProDeckPackCard(name='Alien Hun

## 10. Metadata del DTO

In [27]:
assert pack.external_id == PACK_ID
assert pack.name == EXPECTED_PACK_NAME
assert pack.creator == first["username"]
assert pack.image_id == str(first["img_url"])
assert len(pack.cards) == len(raw)

print("External ID:", pack.external_id)
print("Name:", pack.name)
print("Creator:", pack.creator)
print("Cards:", len(pack.cards))

External ID: 24116
Name: Scaled Invasion
Creator: livius03
Cards: 100


## 11. Prime carte del pack

In [28]:
preview_count = min(10, len(pack.cards))

for card in pack.cards[:preview_count]:
    print(
        f"{card.ydk_id:>8} | "
        f"{card.source_rarity:<15} | "
        f"{card.card_type:<25} | "
        f"{card.name}"
    )

34541863 | Common          | Continuous Spell          | "A" Cell Breeding Device
64163367 | Common          | Continuous Spell          | "A" Cell Incubator
68786330 | Common          | Continuous Spell          | Attack Pheromones
99342953 | Common          | Continuous Spell          | Code A Ancient Ruins
80678380 | Common          | Counter Trap              | Snake Deity's Command
15475415 | Common          | Effect Monster            | Alien Dog
62315111 | Common          | Effect Monster            | Alien Hunter
76573247 | Common          | Effect Monster            | Alien Infiltrator
25920413 | Common          | Effect Monster            | Alien Skull
58066722 | Common          | Effect Monster            | Alien Stealthbuster


## 12. Verifica degli YDK ID

A questo stadio richiediamo soltanto che siano interi positivi.
La risoluzione verso Master Duel verrà implementata successivamente.


In [29]:
invalid_ids = [
    card
    for card in pack.cards
    if not isinstance(card.ydk_id, int) or card.ydk_id <= 0
]

assert not invalid_ids, invalid_ids

print("All YDK IDs are positive integers")

All YDK IDs are positive integers


## 13. Duplicati YDK

Questo test non decide ancora se un duplicato sia legalmente accettabile.
Serve solo a mostrare se il pack sorgente ne contiene.


In [30]:
ydk_counts = Counter(card.ydk_id for card in pack.cards)

duplicates = {
    ydk_id: count
    for ydk_id, count in ydk_counts.items()
    if count > 1
}

print("Unique YDK IDs:", len(ydk_counts))
print("Total rows:", len(pack.cards))
print("Duplicate YDK IDs:", duplicates)

Unique YDK IDs: 100
Total rows: 100
Duplicate YDK IDs: {}


## 14. Rarità sorgente presenti

In [31]:
rarity_counts = Counter(
    card.source_rarity
    for card in pack.cards
)

print("Source rarity distribution:")

for rarity, count in sorted(rarity_counts.items()):
    print(f"  {rarity}: {count}")

Source rarity distribution:
  Common: 48
  Rare: 25
  Secret Rare: 2
  Super Rare: 15
  Ultra Rare: 10


## 15. Tipi di carta presenti

Informazione puramente diagnostica: non viene ancora usata dal domain layer.


In [32]:
type_counts = Counter(
    card.card_type
    for card in pack.cards
)

for card_type, count in sorted(type_counts.items()):
    print(f"{count:>3}  {card_type}")

  5  Continuous Spell
  4  Continuous Trap
  1  Counter Trap
 45  Effect Monster
  1  Equip Spell
  2  Field Spell
  2  Flip Effect Monster
  1  Gemini Effect Monster
  4  Link Effect Monster
  2  Normal Monster
 10  Normal Spell
 12  Normal Trap
  3  Quick-Play Spell
  3  Synchro Effect Monster
  4  Tuner Effect Monster
  1  Xyz Effect Monster


## 16. Controllo specifico di una carta nota

La risposta che avevamo osservato per il pack `24116` conteneva:

```text
"A" Cell Breeding Device
YDK ID: 34541863
```

Questo test aiuta a rilevare cambiamenti sostanziali nel pack sorgente.


In [33]:
known_card = next(
    (
        card
        for card in pack.cards
        if card.ydk_id == 34541863
    ),
    None,
)

assert known_card is not None, (
    "Expected card '\"A\" Cell Breeding Device' was not found."
)

print(known_card)

YgoProDeckPackCard(name='"A" Cell Breeding Device', ydk_id=34541863, card_type='Continuous Spell', source_rarity='Common')
